# Lesson 12: Cursor-Based Pagination & Large Result Sets

In this notebook, you will:
1. Implement a `FastMCP` catalog with Base64 opaque cursor pagination.
2. Write a resilient `paginate_all()` client helper with loop protection.
3. Test `-32602` error handling on malformed cursors.

In [ ]:
!pip install -q -U mcp

In [ ]:
import base64
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("PaginatedCatalog")
ITEMS = [{"id": i, "name": f"Dataset-{i:03d}"} for i in range(1, 51)]

@mcp.tool()
def list_datasets(cursor: str | None = None, page_size: int = 12) -> dict:
    """Return paginated datasets using an opaque base64 cursor."""
    offset = 0
    if cursor:
        payload = json.loads(base64.urlsafe_b64decode(cursor.encode()).decode())
        offset = int(payload["offset"])
    chunk = ITEMS[offset : offset + page_size]
    next_offset = offset + page_size
    next_cur = base64.urlsafe_b64encode(json.dumps({"offset": next_offset}).encode()).decode() if next_offset < len(ITEMS) else None
    return {"items": chunk, "nextCursor": next_cur}

all_rows, cur = [], None
while True:
    page = list_datasets(cursor=cur)
    all_rows.extend(page["items"])
    cur = page["nextCursor"]
    if not cur:
        break
print(f"Paginated {len(all_rows)} datasets cleanly.")

### Challenge Exercise
Add a `category_filter: str | None = None` field inside the Base64 cursor payload and verify that changing the filter mid-pagination raises a `ValueError`.